In [1]:
import os
import csv
import itertools
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
import matplotlib.pyplot as plt

# --------- Configuration ---------
DATA_DIR = r"E:\Learning\UNSW\Term2\9444\group_project\data\split_with_713"
OUTPUT_DIR = r"E:\Learning\UNSW\Term2\9444\group_project\outputs\plot\final\resnet50_base_optimized"
NUM_CLASSES = 39
BATCH_SIZE = 64
NUM_WORKERS = 8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_EPOCHS = 30
WEIGHT_DECAY = 1e-4
LR_HEAD = 1e-3  # learning rate for classification head
PATIENCE = 5  # early-stopping patience (by accuracy)

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

torch.backends.cudnn.benchmark = True

# --------- Data Transforms ---------
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# --------- Dataloaders ---------
def get_dataloaders(data_dir, batch_size, num_workers):
    train_ds = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_transform)
    val_ds = datasets.ImageFolder(os.path.join(data_dir, "val"), transform=val_transform)
    test_ds = datasets.ImageFolder(os.path.join(data_dir, "test"), transform=val_transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader

# --------- Model Setup ---------
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
for name, param in model.named_parameters():
    if not name.startswith("fc."):
        param.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)

# --------- Optimizer & Scheduler ---------
optimizer = AdamW(model.fc.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler("cuda")

# --------- Helper: collect predictions ---------
def evaluate_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.tolist())
    return all_labels, all_preds

# --------- Training Loop ---------
def train():
    print(">> start:", flush=True)
    train_loader, val_loader, test_loader = get_dataloaders(DATA_DIR, BATCH_SIZE, NUM_WORKERS)

    best_val_acc = 0.0
    epochs_no_improve = 0

    epochs_list, train_losses, val_accuracies, val_f1s = [], [], [], []

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with autocast("cuda"):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * imgs.size(0)

        scheduler.step()
        train_loss = running_loss / len(train_loader.dataset)

        y_true, y_pred = evaluate_predictions(model, val_loader)
        val_acc = accuracy_score(y_true, y_pred)
        val_f1 = f1_score(y_true, y_pred, average="macro")

        epochs_list.append(epoch)
        train_losses.append(train_loss)
        val_accuracies.append(val_acc)
        val_f1s.append(val_f1)

        print(f"Epoch {epoch:2d}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | "
              f"Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_resnet50_leaf.pth"))
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"Early stopping at epoch {epoch}.")
                break

    # Final test evaluation
    y_true_test, y_pred_test = evaluate_predictions(model, test_loader)
    test_acc = accuracy_score(y_true_test, y_pred_test)
    test_prec = precision_score(y_true_test, y_pred_test, average="macro")
    test_rec = recall_score(y_true_test, y_pred_test, average="macro")
    test_f1 = f1_score(y_true_test, y_pred_test, average="macro")

    print("\n===== FINAL TEST METRICS =====\n"
          f"Accuracy : {test_acc:.4f}\n"
          f"Precision: {test_prec:.4f}\n"
          f"Recall   : {test_rec:.4f}\n"
          f"F1-Score : {test_f1:.4f}\n")

    # Confusion matrix with matplotlib
    cm = confusion_matrix(y_true_test, y_pred_test)
    plt.figure(figsize=(12, 10))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title("Confusion Matrix — Test Set")
    plt.colorbar(shrink=0.8)
    tick_marks = range(NUM_CLASSES)
    plt.xticks(tick_marks)
    plt.yticks(tick_marks)
    thresh = cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black")
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix_test.png"), dpi=300)
    plt.close()

    # Save metrics to CSV
    metrics_path = os.path.join(OUTPUT_DIR, "metrics.csv")
    with open(metrics_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "val_acc", "val_f1"])
        for e, l, a, f1 in zip(epochs_list, train_losses, val_accuracies, val_f1s):
            writer.writerow([e, l, a, f1])

    # Plot learning curves
    # Training Loss
    plt.figure()
    plt.plot(epochs_list, train_losses, marker="o")
    plt.title("Training Loss vs. Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_loss.png"), dpi=300)
    plt.close()

    # Validation Accuracy
    plt.figure()
    plt.plot(epochs_list, val_accuracies, marker="o")
    plt.title("Validation Accuracy vs. Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "validation_accuracy.png"), dpi=300)
    plt.close()

    # Validation F1-Score
    plt.figure()
    plt.plot(epochs_list, val_f1s, marker="o")
    plt.title("Validation F1-Score vs. Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("F1-Score")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "validation_f1.png"), dpi=300)
    plt.close()

if __name__ == "__main__":
    train()


>> start:
Epoch  1/30 | Train Loss: 1.2484 | Val Acc: 0.8451 | Val F1: 0.8268
Epoch  2/30 | Train Loss: 0.6254 | Val Acc: 0.8788 | Val F1: 0.8632
Epoch  3/30 | Train Loss: 0.5084 | Val Acc: 0.9030 | Val F1: 0.8912
Epoch  4/30 | Train Loss: 0.4458 | Val Acc: 0.9018 | Val F1: 0.8900
Epoch  5/30 | Train Loss: 0.4120 | Val Acc: 0.9076 | Val F1: 0.8972
Epoch  6/30 | Train Loss: 0.3866 | Val Acc: 0.9095 | Val F1: 0.8983
Epoch  7/30 | Train Loss: 0.3713 | Val Acc: 0.9166 | Val F1: 0.9077
Epoch  8/30 | Train Loss: 0.3573 | Val Acc: 0.9136 | Val F1: 0.9045
Epoch  9/30 | Train Loss: 0.3444 | Val Acc: 0.9232 | Val F1: 0.9154
Epoch 10/30 | Train Loss: 0.3302 | Val Acc: 0.9220 | Val F1: 0.9126
Epoch 11/30 | Train Loss: 0.3210 | Val Acc: 0.9244 | Val F1: 0.9155
Epoch 12/30 | Train Loss: 0.3162 | Val Acc: 0.9257 | Val F1: 0.9172
Epoch 13/30 | Train Loss: 0.3154 | Val Acc: 0.9269 | Val F1: 0.9196
Epoch 14/30 | Train Loss: 0.2999 | Val Acc: 0.9321 | Val F1: 0.9243
Epoch 15/30 | Train Loss: 0.2992 | Val